# Wan2.1-T2V-1.3B Prompt Gain Diagnostics Run-All

This notebook is analysis-only. It does not run `generate.py`, does not load Wan weights, and does not create new videos.

Use Colab `Runtime > Run all` after the Wan2.1 BSS/BDS experiment has produced metrics or videos. It restores the experiment folder from Drive, optionally recomputes metrics from existing videos if the metrics CSV is missing, then writes:

- `tables/per_prompt_gain_table.csv`
- `tables/prompt_attribute_gain_summary.csv`
- `figures/gain_waterfall_10_20_30_40.png`
- `figures/side_by_side/failure_prompt_gallery.html`
- `reports/04_prompt_gain_diagnostics.md`

In [ ]:
from pathlib import Path
import os


REPO_URL = "https://github.com/WANG-Ruipeng/Wan2.1.git"
BRANCH = "main"
REPO_DIR = Path("/content/Wan2.1")

EXPERIMENT_NAME = "wan21_13b_bss_bds_v1"
LOCAL_EXP_ROOT = REPO_DIR / "bss_experiments" / EXPERIMENT_NAME
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS")
DRIVE_EXP_ROOT = DRIVE_PROJECT_ROOT / EXPERIMENT_NAME

MOUNT_DRIVE = True
FORCE_RECLONE = False
INSTALL_ANALYSIS_DEPS = True
RESTORE_FROM_DRIVE = True
RESTORE_HEAVY_VIDEOS = True  # needed for gallery videos; set False for tables/plots only
RUN_METRICS_IF_MISSING = True
SYNC_ANALYSIS_TO_DRIVE = True

print("Repo:", REPO_URL, "branch", BRANCH)
print("Local experiment:", LOCAL_EXP_ROOT)
print("Drive experiment:", DRIVE_EXP_ROOT)

Repo: https://github.com/WANG-Ruipeng/Wan2.1.git branch main
Local experiment: /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1
Drive experiment: /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/wan21_13b_bss_bds_v1


In [11]:
import getpass
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped or failed:", repr(exc))

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def display_cmd(cmd):
    text = " ".join(str(part) for part in cmd)
    if "x-access-token:" in text:
        text = text.split("x-access-token:")[0] + "x-access-token:***@" + text.split("@", 1)[-1]
    return text


def run(cmd, cwd=None, check=True):
    print("$", display_cmd(cmd))
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True)


def run_capture(cmd, cwd=None, check=False):
    print("$", display_cmd(cmd))
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)


def authenticated_url(url):
    token = os.environ.get("GITHUB_TOKEN")
    if token is None:
        token = getpass.getpass("GitHub token for clone/fetch; leave blank if the branch is public: ")
        if token:
            os.environ["GITHUB_TOKEN"] = token
    if token:
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url


def rsync_or_copy(src, dst, exclude_videos=False):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        cmd = ["rsync", "-a", str(src) + "/", str(dst) + "/"]
        if exclude_videos:
            cmd[2:2] = ["--exclude", "outputs/videos/"]
        run(cmd)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
clone_url = authenticated_url(REPO_URL)

if FORCE_RECLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    fallback = REPO_DIR.with_name(f"{REPO_DIR.name}_non_git_{int(time.time())}")
    print(f"Existing non-git directory found at {REPO_DIR}; moving it to {fallback}")
    shutil.move(str(REPO_DIR), str(fallback))

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", clone_url, str(REPO_DIR)])
else:
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", clone_url])
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
    switched = run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=False)
    if switched.returncode != 0:
        run(["git", "-C", str(REPO_DIR), "switch", "--track", "-c", BRANCH, f"origin/{BRANCH}"])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH])

print(run_capture(["git", "-C", str(REPO_DIR), "status", "--short"]).stdout)
print(run_capture(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"]).stdout)

REQUIRED_REPO_FILES = [
    "scripts/analyze_prompt_gain_diagnostics.py",
    "scripts/compute_metrics_against_ref.py",
]
missing_repo_files = [name for name in REQUIRED_REPO_FILES if not (REPO_DIR / name).exists()]
if missing_repo_files:
    raise FileNotFoundError(
        "The cloned branch does not contain the analysis scripts. Push the local changes to this branch, then rerun. Missing: "
        + ", ".join(missing_repo_files)
    )
print("Required analysis scripts are present.")

$ git -C /content/Wan2.1 remote set-url origin https://github.com/WANG-Ruipeng/Wan2.1.git
$ git -C /content/Wan2.1 fetch origin main
$ git -C /content/Wan2.1 switch main
$ git -C /content/Wan2.1 pull --ff-only origin main
$ git -C /content/Wan2.1 status --short
 M bss_experiments/wan21_13b_bss_bds_v1/manifests/wan21_13b_prompt_suite_manifest.csv
 M bss_experiments/wan21_13b_bss_bds_v1/manifests/wan21_13b_smoke_manifest.csv
 M bss_experiments/wan21_13b_bss_bds_v1/metrics/schedule_validation_summary.csv
 M bss_experiments/wan21_13b_bss_bds_v1/reports/00_repo_model_hardware_audit.md
 M bss_experiments/wan21_13b_bss_bds_v1/reports/00b_wan22_future_extension_audit.md
 M bss_experiments/wan21_13b_bss_bds_v1/reports/01_wan21_13b_smoke_report.md
 M bss_experiments/wan21_13b_bss_bds_v1/reports/02_mini_suite_run_report.md
 M bss_experiments/wan21_13b_bss_bds_v1/reports/03_bds_report.md
 M bss_experiments/wan21_13b_bss_bds_v1/reports/FINAL_WAN21_13B_BSS_BDS_REPORT.md
 M bss_experiments/wan21_13b_

In [13]:
if RESTORE_FROM_DRIVE:
    if not DRIVE_EXP_ROOT.exists():
        raise FileNotFoundError(f"Drive experiment folder not found: {DRIVE_EXP_ROOT}")
    print(f"Restoring experiment artifacts: {DRIVE_EXP_ROOT} -> {LOCAL_EXP_ROOT}")
    rsync_or_copy(DRIVE_EXP_ROOT, LOCAL_EXP_ROOT, exclude_videos=not RESTORE_HEAVY_VIDEOS)
else:
    print("RESTORE_FROM_DRIVE=False; using existing local experiment folder.")

if not LOCAL_EXP_ROOT.exists():
    raise FileNotFoundError(f"Local experiment folder not found: {LOCAL_EXP_ROOT}")

print("Local experiment ready:", LOCAL_EXP_ROOT)
print("Existing top-level files:")
run(["find", str(LOCAL_EXP_ROOT), "-maxdepth", "2", "-type", "f"], check=False)

Restoring experiment artifacts: /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/wan21_13b_bss_bds_v1 -> /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1
$ rsync -a /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/wan21_13b_bss_bds_v1/ /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/
Local experiment ready: /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1
Existing top-level files:
$ find /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1 -maxdepth 2 -type f


CompletedProcess(args=['find', '/content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1', '-maxdepth', '2', '-type', 'f'], returncode=0)

In [14]:
if INSTALL_ANALYSIS_DEPS:
    run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "imageio", "imageio-ffmpeg"])
    print("Analysis dependencies installed/checked.")
else:
    print("INSTALL_ANALYSIS_DEPS=False; skipping dependency install.")

$ /usr/bin/python3 -m pip install -q numpy matplotlib imageio imageio-ffmpeg
Analysis dependencies installed/checked.


In [15]:
metrics_dir = LOCAL_EXP_ROOT / "metrics"
gain_csv = metrics_dir / "same_compute_gain_long.csv"
master_csv = metrics_dir / "master_long_metrics.csv"
full_manifest = LOCAL_EXP_ROOT / "manifests/wan21_13b_prompt_suite_manifest.csv"
smoke_manifest = LOCAL_EXP_ROOT / "manifests/wan21_13b_smoke_manifest.csv"

need_metrics = (not gain_csv.exists()) or gain_csv.stat().st_size == 0 or (not master_csv.exists()) or master_csv.stat().st_size == 0
if need_metrics:
    if not RUN_METRICS_IF_MISSING:
        raise FileNotFoundError(f"Metrics missing and RUN_METRICS_IF_MISSING=False: {gain_csv}")
    manifest = full_manifest if full_manifest.exists() else smoke_manifest
    if not manifest.exists():
        raise FileNotFoundError("No manifest found to recompute metrics.")
    print("Metrics missing; recomputing from existing videos using manifest:", manifest)
    run([sys.executable, str(REPO_DIR / "scripts/compute_metrics_against_ref.py"),
         "--repo_dir", str(REPO_DIR),
         "--manifest", str(manifest),
         "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR))])
else:
    print("Existing metrics found; skipping metric recompute.")

print("Metrics status:")
for p in [master_csv, gain_csv]:
    print(p, "OK" if p.exists() and p.stat().st_size else "missing/empty")

Existing metrics found; skipping metric recompute.
Metrics status:
/content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/metrics/master_long_metrics.csv OK
/content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/metrics/same_compute_gain_long.csv OK


In [16]:
run([sys.executable, str(REPO_DIR / "scripts/analyze_prompt_gain_diagnostics.py"),
     "--repo_dir", str(REPO_DIR),
     "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR))])

$ /usr/bin/python3 /content/Wan2.1/scripts/analyze_prompt_gain_diagnostics.py --repo_dir /content/Wan2.1 --output_root bss_experiments/wan21_13b_bss_bds_v1


CompletedProcess(args=['/usr/bin/python3', '/content/Wan2.1/scripts/analyze_prompt_gain_diagnostics.py', '--repo_dir', '/content/Wan2.1', '--output_root', 'bss_experiments/wan21_13b_bss_bds_v1'], returncode=0)

In [17]:
if SYNC_ANALYSIS_TO_DRIVE:
    DRIVE_EXP_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Syncing analysis outputs to Drive: {LOCAL_EXP_ROOT} -> {DRIVE_EXP_ROOT}")
    # Keep the sync light; original videos should already be in Drive from the main run.
    rsync_or_copy(LOCAL_EXP_ROOT, DRIVE_EXP_ROOT, exclude_videos=True)
else:
    print("SYNC_ANALYSIS_TO_DRIVE=False; skipping Drive sync.")

Syncing analysis outputs to Drive: /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1 -> /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/wan21_13b_bss_bds_v1
$ rsync -a --exclude outputs/videos/ /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/ /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/wan21_13b_bss_bds_v1/


In [18]:
important_paths = {
    "per-prompt gains": LOCAL_EXP_ROOT / "tables/per_prompt_gain_table.csv",
    "per-prompt gains md": LOCAL_EXP_ROOT / "tables/per_prompt_gain_table.md",
    "attribute gains": LOCAL_EXP_ROOT / "tables/prompt_attribute_gain_summary.csv",
    "attribute gains md": LOCAL_EXP_ROOT / "tables/prompt_attribute_gain_summary.md",
    "gain waterfall": LOCAL_EXP_ROOT / "figures/gain_waterfall_10_20_30_40.png",
    "failure gallery": LOCAL_EXP_ROOT / "figures/side_by_side/failure_prompt_gallery.html",
    "diagnostics report": LOCAL_EXP_ROOT / "reports/04_prompt_gain_diagnostics.md",
    "Drive mirror": DRIVE_EXP_ROOT,
}

for label, path in important_paths.items():
    print(f"{label:22s} {'OK' if path.exists() else 'not yet'}  {path}")

report = LOCAL_EXP_ROOT / "reports/04_prompt_gain_diagnostics.md"
if report.exists():
    print("\n===== Prompt Gain Diagnostics Report =====")
    print(report.read_text(encoding="utf-8"))

per_prompt = LOCAL_EXP_ROOT / "tables/per_prompt_gain_table.csv"
if per_prompt.exists():
    print("\n===== Per-Prompt Gain Table Preview =====")
    print("\n".join(per_prompt.read_text(encoding="utf-8").splitlines()[:12]))

per-prompt gains       OK  /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/tables/per_prompt_gain_table.csv
per-prompt gains md    OK  /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/tables/per_prompt_gain_table.md
attribute gains        OK  /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/tables/prompt_attribute_gain_summary.csv
attribute gains md     OK  /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/tables/prompt_attribute_gain_summary.md
gain waterfall         OK  /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/figures/gain_waterfall_10_20_30_40.png
failure gallery        OK  /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/figures/side_by_side/failure_prompt_gallery.html
diagnostics report     OK  /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/reports/04_prompt_gain_diagnostics.md
Drive mirror           OK  /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/wan21_13b_bss_bds_v1

===== Prompt Gain Diagnostics Report =====
# Prompt Gain Diagnosti